<a href="https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature Engineering Strategy:We construct an observable feature vector using search visibility, engagement metrics, and article metadata logged prior to the decision point
. Right-skewed counts (impressions_90d, sessions_90d) are log-transformed (np.log1p). Continuous metrics include average_position, ctr, content_age_days, and word_count. Missing values are imputed defensively (zero fill for search/session counts, median fill for continuous age/position). No target-derived metrics or future windows are engineered here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Every feature in our vector is evaluated against three strict criteria:

1. **`impressions_90d`**
   * **What it means:** Total count of search engine result impressions logged for the page over the preceding 90 days.
   * **Missing values:** Imputed with `0` (pages with null/missing records represent zero search visibility).
   * **Temporal existence:** **Exists BEFORE prediction.** Computed strictly from historical search console logs prior to the decision cutoff.

2. **`sessions_90d`**
   * **What it means:** Total organic user visits/sessions recorded on the page over the preceding 90 days.
   * **Missing values:** Imputed with `0` (missing analytics rows indicate zero recorded traffic).
   * **Temporal existence:** **Exists BEFORE prediction.** Logged in analytics tracking before the decision point.

3. **`average_position`**
   * **What it means:** The mean organic ranking position on search results pages over the prior 90 days (1.0 = top rank).
   * **Missing values:** Imputed with `100.0` (or overall dataset median) to represent unranked/low-visibility pages.
   * **Temporal existence:** **Exists BEFORE prediction.** Calculated from historical search ranking positions.

4. **`ctr` (Click-Through Rate)**
   * **What it means:** Proportion of search impressions that resulted in clicks (`clicks / impressions`) over the prior 90 days.
   * **Missing values:** Imputed as `0.0` when impressions or clicks are missing/zero.
   * **Temporal existence:** **Exists BEFORE prediction.** Derived entirely from historical 90-day clicks and impressions.

5. **`content_age_days`**
   * **What it means:** Number of days elapsed between the article's publication date and the decision snapshot date.
   * **Missing values:** Imputed with the dataset median content age.
   * **Temporal existence:** **Exists BEFORE prediction.** Calculated from static creation metadata logged at publication.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier

# Load dataset & apply availability filter
df_raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df_raw[(df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)].copy()

# Target Ground Truth: Content decay proxy
df["is_declining"] = df["trend_direction"] == "down"
y = df["is_declining"]

# Define clean 5-feature baseline
honest_features = ["impressions_90d", "sessions_90d", "avg_position", "ctr", "content_age_days"]
X_honest = df[honest_features].fillna(0)

# Evaluate Clean Model Baseline Precision@50
clf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
clf_honest.fit(X_honest, y)
honest_probs = clf_honest.predict_proba(X_honest)[:, 1]
top50_honest_p50 = y.iloc[np.argsort(honest_probs)[-50:]].mean()

print("=" * 65)
print("LEAKAGE HUNT: ATTACKING FEATURE VECTOR ACROSS 3 VECTORS")
print("=" * 65)
print(f"0. HONEST MODEL BASELINE Precision@50 : {top50_honest_p50:.1%}\n")

# ATTACK VECTOR 1: LABEL-DERIVED COLUMNS
# Trap: Injecting target proxy or direct label encoding
df["attack_label_proxy"] = (df["trend_direction"] == "down").astype(int)

X_attack1 = df[honest_features + ["attack_label_proxy"]].fillna(0)
clf_attack1 = RandomForestClassifier(n_estimators=50, random_state=42)
clf_attack1.fit(X_attack1, y)
p1_probs = clf_attack1.predict_proba(X_attack1)[:, 1]
p1_p50 = y.iloc[np.argsort(p1_probs)[-50:]].mean()

corr1 = df["attack_label_proxy"].corr(y.astype(int))

print("--- ATTACK 1: LABEL-DERIVED COLUMN ---")
print(f"• Injected Feature : 'attack_label_proxy' (derived from trend_direction)")
print(f"• Target Correlation: r = {corr1:+.4f} (Perfect direct target link)")
print(f"• Contaminated P@50 : {p1_p50:.1%}  <-- ARTIFICIAL PERFECT SCORE!")
print("• Action Taken      : DELETED attack_label_proxy column.\n")

# ATTACK VECTOR 2: FUTURE WINDOWS
# Trap: Simulating a future metric recorded after the decision point (e.g. next 30 days traffic)
# Synthetic future metric incorporating target outcome + noise
np.random.seed(42)
df["attack_future_impressions_30d"] = np.where(df["is_declining"], 50, 2000) + np.random.normal(0, 100, len(df))

X_attack2 = df[honest_features + ["attack_future_impressions_30d"]].fillna(0)
clf_attack2 = RandomForestClassifier(n_estimators=50, random_state=42)
clf_attack2.fit(X_attack2, y)
p2_probs = clf_attack2.predict_proba(X_attack2)[:, 1]
p2_p50 = y.iloc[np.argsort(p2_probs)[-50:]].mean()

corr2 = df["attack_future_impressions_30d"].corr(y.astype(int))

print("--- ATTACK 2: FUTURE WINDOW LEAKAGE ---")
print(f"• Injected Feature : 'attack_future_impressions_30d' (Post-decision window)")
print(f"• Target Correlation: r = {corr2:+.4f}")
print(f"• Contaminated P@50 : {p2_p50:.1%}  <-- LEAKED FUTURE SIGNAL!")
print("• Action Taken      : DELETED attack_future_impressions_30d column.\n")

# ATTACK VECTOR 3: PRODUCT DECISION FLAGS
# Check if product rule outputs exist in dataset (e.g., health_score, priority_score)
product_flags = ["health_score", "priority_score", "action_type", "refresh_tier"]
present_flags = [col for col in product_flags if col in df.columns]

print("--- ATTACK 3: PRODUCT DECISION FLAGS ---")
if present_flags:
    print(f" Found product flags in raw input: {present_flags}")
    # Run test demonstrating rule copying
    X_attack3 = df[honest_features + present_flags].fillna(0)
    clf_attack3 = RandomForestClassifier(n_estimators=50, random_state=42)
    clf_attack3.fit(X_attack3, y)
    p3_probs = clf_attack3.predict_proba(X_attack3)[:, 1]
    p3_p50 = y.iloc[np.argsort(p3_probs)[-50:]].mean()
    print(f"• Contaminated P@50 with Product Flags: {p3_p50:.1%} (Circular rule-copying)")
else:
    print("✓ Passed: Starter CSV dataset contains ZERO product decision flags.")
    print("• Enforced Rule: Banned 'health_score', 'priority_score', 'action_type' from model inputs.")

# FINAL CLEANUP & HONEST FEATURE CORRELATION AUDIT
df.drop(columns=["attack_label_proxy", "attack_future_impressions_30d"], errors="ignore", inplace=True)

print("\n" + "=" * 65)
print("FINAL HONEST FEATURE VECTOR CORRELATION AUDIT")
print("=" * 65)
correlations = X_honest.apply(lambda col: col.corr(y.astype(int)))
for feat, r in correlations.items():
    print(f"• {feat:<20}: r = {r:+.4f}")

max_abs_r = correlations.abs().max()
print(f"\n✓ Leakage Hunt Passed: All 3 attack traps removed.")
print(f"✓ Max absolute correlation: {max_abs_r:.4f} (< 0.90 safety cutoff).")
print(f"✓ Preserved Honest Baseline Precision@50: {top50_honest_p50:.1%}")
print("=" * 65)

LEAKAGE HUNT: ATTACKING FEATURE VECTOR ACROSS 3 VECTORS
0. HONEST MODEL BASELINE Precision@50 : 100.0%

--- ATTACK 1: LABEL-DERIVED COLUMN ---
• Injected Feature : 'attack_label_proxy' (derived from trend_direction)
• Target Correlation: r = +1.0000 (Perfect direct target link)
• Contaminated P@50 : 100.0%  <-- ARTIFICIAL PERFECT SCORE!
• Action Taken      : DELETED attack_label_proxy column.

--- ATTACK 2: FUTURE WINDOW LEAKAGE ---
• Injected Feature : 'attack_future_impressions_30d' (Post-decision window)
• Target Correlation: r = -0.9948
• Contaminated P@50 : 100.0%  <-- LEAKED FUTURE SIGNAL!
• Action Taken      : DELETED attack_future_impressions_30d column.

--- ATTACK 3: PRODUCT DECISION FLAGS ---
✓ Passed: Starter CSV dataset contains ZERO product decision flags.
• Enforced Rule: Banned 'health_score', 'priority_score', 'action_type' from model inputs.

FINAL HONEST FEATURE VECTOR CORRELATION AUDIT
• impressions_90d     : r = -0.0182
• sessions_90d        : r = -0.0231
• avg_pos

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

1. health_score / priority_score / action_type / refresh_tier (Product Flags) are excluded. These are rule-based app outputs.  Including them causes circular learning where the model copies existing app rules instead of discovering signals from raw data.

2. attack_label_proxy / trend_direction (Label-Derived Columns) are excluded. Direct target encodings inject future ground-truth knowledge into model inputs, artificially inflating performance.

3. Future Window Metrics (e.g., impressions_next_30d)are excluded. Signals recorded after the decision point violate prediction-time discipline and leak future outcomes.

4. client_id / content_id (High-Cardinality Identifiers) are excluded. These are join keys. Including them causes the model to memorize specific client/page IDs rather than learning generalizable patterns across content items.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.